# M2 Notebook 22 — Forecasting Methods

**Status:** Runnable first edition

## Learning objectives

- Apply seasonal naive, exponential smoothing, and autoregression.
- Evaluate forecasts through walk-forward validation.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    AutoregressiveModel,ExponentialSmoothing,mean_absolute_scaled_error,
    seasonal_naive_forecast,walk_forward_validate,
)


In [ ]:
rng=np.random.default_rng(22)
t=np.arange(180)
series=50+5*np.sin(2*np.pi*t/12)+0.1*t+rng.normal(scale=1.5,size=len(t))
train=series[:-24]; test=series[-24:]
forecasts={
    "Seasonal naive":seasonal_naive_forecast(train,24,12),
    "Exp smoothing":ExponentialSmoothing(.3).fit(train).forecast(24),
    "AR(12)":AutoregressiveModel(12).fit(train).forecast(24),
}
pd.Series({name:mean_absolute_scaled_error(test,pred,train,12) for name,pred in forecasts.items()})


In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(np.arange(len(train),len(series)),test,label="actual")
for name,pred in forecasts.items():
    ax.plot(np.arange(len(train),len(series)),pred,label=name)
ax.legend(); ax.set_title("Forecast Comparison")
plt.show()


## Walk-forward validation

In [ ]:
actual,pred=walk_forward_validate(lambda:AutoregressiveModel(12),series,initial_train=96,horizon=12)
{"MASE":mean_absolute_scaled_error(actual,pred,series[:96],12)}


## Decision Intelligence case

Forecast accuracy should be assessed at the same horizon and update frequency used operationally.

## Key insight

A strong forecasting baseline and honest walk-forward validation are more valuable than a complex model evaluated incorrectly.